In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow version:", tf.__version__)


In cmd , dowloaded the dataset ---> kaggle datasets download -d emmarex/plantdisease -p part2_plant_disease\data --unzip

In [ ]:
DATASET_PATH = "../part2_plant_disease/data/PlantVillage"
IMAGE_SIZE = (128, 128)             
BATCH_SIZE = 32                     

datagen = ImageDataGenerator(rescale=1./255, validation_split=0.20)               #rescale for normalization

train_data = datagen.flow_from_directory(
    DATASET_PATH, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", subset="training", shuffle=True)

validation_data = datagen.flow_from_directory(
    DATASET_PATH, target_size=IMAGE_SIZE, batch_size=BATCH_SIZE,
    class_mode="categorical", subset="validation", shuffle=False)

print("Number of classes:", train_data.num_classes)
print(list(train_data.class_indices.keys()))


In [ ]:
model = Sequential([
    Conv2D(32 , (3,3), activation="relu", input_shape=(128,128,3)),
    MaxPooling2D(2,2),           #This reduces the size of the feature maps.
    Conv2D(64, (3,3), activation="relu"),  # relu {non linearity} , learns more complex patterns 
    MaxPooling2D(2,2),       
    Dense(128, activation="relu"),   #128 neurons in the dense layer 
    Dropout(0.5),
    Dense(train_data.num_classes, activation="softmax")
])

model.compile(optimizer="adam",              #Adam controls how the model's weights are updated during training.
              loss="categorical_crossentropy",
              metrics=["accuracy"])
model.summary()


In [ ]:
history = model.fit(
    train_data,
    validation_data=validation_data,
    epochs=10
)                                                  #TRAINING DATA


## Evaluation


In [ ]:
loss, accuracy = model.evaluate(validation_data)
print("Validation Accuracy:", round(accuracy*100, 2), "%")

validation_data.reset()
predictions = model.predict(validation_data)
y_pred = np.argmax(predictions, axis=1)
y_true = validation_data.classes
class_names = list(validation_data.class_indices.keys())

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))


In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12,10))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Plant Disease Detection - Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()
